# Prompts

1. Prompt template
2. Chat prompt template 
    - messages 
    - AI/Human/System prompt templates
    - messagePlaceHolder
3. FewShot prompt template
4. Fewshot Chat message prompt
5. Dynamic example selection -> need embedding and vector store knowlwdge

Using - openai/gpt-oss-20b

# LLM 

In [ ]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm_groq = ChatGroq(model="openai/gpt-oss-20b")

# print(llm_groq.invoke("What is softmax?"))

# 1. Prompt template


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_temp = PromptTemplate.from_template("My name is {name}, age is {age} and work in {company} from branch {branch}")
prompt_temp = prompt_temp.partial(branch = "CS")
# temp_ans = prompt_temp.invoke({"name":"vsh","age":"30","company":"ABC"})
# print(temp_ans)

# temp_ans = prompt_temp.format(name = "vsh",age = "34",company = "SDF")
# print(temp_ans)

chain = prompt_temp | llm_groq

answer = chain.invoke({"name":"vsh","age":"30","company":"ABC"})
# print(answer)
print(answer.content)

# Chat prompt template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

messages = [
    ("system","You are an assistant and your anme is {name}"),
    ("human","Hi, how are you doing?"),
    ("ai","Doing good, how can I help you?"),
    ("human","what is attention mechanism?")
    # ("palceholder","{history}")
]

chat_template = ChatPromptTemplate.from_messages(messages = messages)
# chat_ans = chat_template.invoke({"name":"ai assistant"})

#llm 

chain = chat_template | llm_groq
answer = chain.invoke({"name":"ai assistant"})
print(answer.content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

messages1 = [
    ("human","Who is Virat Kohli?")
]

chat_template1 = ChatPromptTemplate.from_messages(messages = messages1)

chain = chat_template1 | llm_groq
answer = chain.invoke({})
print(answer.content)

messages1.append(answer)
messages2 = [
    ("system","You are designed for text generation."),
    ("human","{anotherQuestion}"),
    MessagesPlaceholder(variable_name="history")
]

chat_template2 = ChatPromptTemplate.from_messages(messages = messages2)
chain2 = chat_template2 | llm_groq
answer = chain2.invoke({"anotherQuestion":"Who is his wife?","history":messages1})
print(answer.content)

#FewShotPromptTemplate

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example = [
    {"input":"on","output":"off"},
    {"input":"short","output":"tall"},
    {"input":"up","output":"down"}
]

base_template = PromptTemplate.from_template("input:{input}\noutput:{output}")

fewShot_template = FewShotPromptTemplate(examples=example, example_prompt=base_template, 
                                         prefix= "Give me the opposite words",suffix="input:{adjective}\noutput:",input_variables=["adjective"])

template = fewShot_template.invoke({"adjective":"bright"})
print(template.text)

chain = fewShot_template | llm_groq
answer = chain.invoke({"adjective":"bright"})
print(answer.text)



# FewShotMessagePromptTempalte

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate , ChatPromptTemplate

example = [
    {"input":"on","output":"off"},
    {"input":"short","output":"tall"},
    {"input":"up","output":"down"}
]

base_template = ChatPromptTemplate.from_messages([
    ("human","{input}"),
    ("ai","{output}")
])

fewshot_promptTemplate = FewShotChatMessagePromptTemplate(examples = example, example_prompt=base_template)

final_template = ChatPromptTemplate.from_messages([
    ('system',"Designed for giving opposite words"),
    fewshot_promptTemplate, 
    ('human','{adjective}')
])

print(final_template.invoke({"adjective":"bright"}).to_messages())

chain = final_template | llm_groq
answer = chain.invoke({"adjective":"bright"})
print(answer)